**IMPORT PANDAS LIBRARY**

In [1]:
import pandas as pd

**IMPORT GDP BY LOCAL AUTHORITIES FILE** 

In [2]:
df_gdp = pd.read_excel("../data/raw/regionalgrossdomesticproductgdplocalauthorities.xlsx", sheet_name='Table 7', header=1)

df_gdp.head(3)

,ITL1 Region,LA code,LA name,1998,1999,2000,2001,2002,2003,2004,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,North East,E06000001,Hartlepool,10003,10577,10775,10854,11468,12071,12544,...,15520,16371,17674,17944,17649,18090,18300,17912,18853,20154
1,North East,E06000004,Stockton-on-Tees,15842,16434,16793,16947,17748,18943,20406,...,26158,27628,28028,27682,28169,29771,29025,30906,33034,33314
2,North East,E06000002,Middlesbrough,11825,12431,12860,13117,13736,14668,15894,...,20376,21142,21901,23035,24503,24429,22388,24927,26941,28066


**SLICE 2023 GDP PER CAPITA VALUES ONLY**

In [3]:
df_gdp_23 = df_gdp.loc[:, ['ITL1 Region', 'LA code', 'LA name', '2023']]

df_gdp_23.head(3)

,ITL1 Region,LA code,LA name,2023
0,North East,E06000001,Hartlepool,20154
1,North East,E06000004,Stockton-on-Tees,33314
2,North East,E06000002,Middlesbrough,28066


**CONFIRM THERE ARE NO NaNs IN 2023 SLICE**

In [4]:
df_gdp_23.isna().sum()

ITL1 Region    0
LA code        0
LA name        0
2023           0
dtype: int64

**NORTHERN IRLENAD HAS NO GDP PER CAPITA DATA FOR 2023, BOOLEAN MASK TO CONFIRM THIS**

In [5]:
df_gdp_23.loc[df_gdp_23['ITL1 Region'] == 'Northern Ireland']

,ITL1 Region,LA code,LA name,2023
350,Northern Ireland,N09000003,Belfast,[u]
351,Northern Ireland,N09000002,"Armagh City, Banbridge and Craigavon",[u]
352,Northern Ireland,N09000010,"Newry, Mourne and Down",[u]
353,Northern Ireland,N09000011,Ards and North Down,[u]
354,Northern Ireland,N09000005,Derry City and Strabane,[u]
355,Northern Ireland,N09000009,Mid Ulster,[u]
356,Northern Ireland,N09000004,Causeway Coast and Glens,[u]
357,Northern Ireland,N09000001,Antrim and Newtownabbey,[u]
358,Northern Ireland,N09000007,Lisburn and Castlereagh,[u]
359,Northern Ireland,N09000008,Mid and East Antrim,[u]


**SLICE NORTHERN IRELAND OFF**

In [6]:
df_gdp_23 = df_gdp_23.loc[df_gdp_23['2023'] != '[u]', :]

df_gdp_23.tail()

,ITL1 Region,LA code,LA name,2023
345,Scotland,S12000026,Scottish Borders,27485
346,Scotland,S12000006,Dumfries and Galloway,32027
347,Scotland,S12000008,East Ayrshire,23015
348,Scotland,S12000028,South Ayrshire,30057
349,Scotland,S12000029,South Lanarkshire,28895


**RENAME COLUMNS ACCORDING TO PERSONAL STYLE**

In [7]:
df_gdp_23 = df_gdp_23.rename(columns={'ITL1 Region':'ITL1_region', 'LA code':'gss_ons_code', 'LA name':'local_authority'})

df_gdp_23.head(2)

,ITL1_region,gss_ons_code,local_authority,2023
0,North East,E06000001,Hartlepool,20154
1,North East,E06000004,Stockton-on-Tees,33314


**CONFIRM DATATYPES**

In [8]:
df_gdp_23.dtypes

ITL1_region           str
gss_ons_code          str
local_authority       str
2023               object
dtype: object

**CHANGE 2023 DTYPES FOR NUMERIC**

In [9]:
df_gdp_23['2023'] = pd.to_numeric(df_gdp_23['2023'])

df_gdp_23.dtypes

ITL1_region          str
gss_ons_code         str
local_authority      str
2023               int64
dtype: object

**SORT LOCAL AUTHORITIES BY GDP PER CAPITA: HIGHEST TO LOWEST**

In [10]:
df_gdp_23 = df_gdp_23.sort_values(by='2023', ascending=False)

df_gdp_23.head(3)

,ITL1_region,gss_ons_code,local_authority,2023
175,London,E09000001,City of London,8228049
176,London,E09000033,Westminster,459185
177,London,E09000007,Camden,188289


**EXPORT FILE TO DATA/PROCESSED**

In [11]:
excel_path = ("../data/processed/uk_local_economy.xlsx")

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl",
    mode='w'
) as writer:

    df_gdp_23.to_excel(writer, sheet_name='gdp_per_capita_2023', index=False)